# 08 Final Research Summary Packet
Question: What is the concise final research summary packet after consolidating the real-macro integration, regime diagnostics, and transition diagnostics work?

**Warning:** This notebook is research-only and not investment advice. It must not be used for forecasting, signal generation, ranking, optimization, allocation, dashboarding, or trading decisions.

This notebook consolidates the descriptive outputs from Notebooks 05-07 into a compact, table-oriented review packet using the checked-in real Part I macro files and the current strategy return input.


## Load Real Inputs And Reuse Locked Validations
Load the real macro regime file, the real transition table, and the current strategy return input. Reuse the existing validation path before assembling the final summary packet. Optional strategy metadata fields remain optional; if they are absent, the packet leaves those columns blank rather than failing.


In [ ]:
from pathlib import Path
import sys

def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing 'src' and 'data'.")

REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.conditioning.final_summary_report import build_notebook_08_tables
from src.data.loaders import load_regimes, load_strategy_returns, load_transitions
from src.data.validators import run_all_validations

regimes = load_regimes(REPO_ROOT / "data" / "macro" / "regimes.csv")
transitions = load_transitions(REPO_ROOT / "data" / "macro" / "transitions.csv")
strategy_returns = load_strategy_returns(REPO_ROOT / "data" / "strategies" / "strategy_returns_example.csv")

run_all_validations(regimes, transitions, strategy_returns)

tables = build_notebook_08_tables(regimes, transitions, strategy_returns, sparse_threshold=3, window=3)
display(tables["input_coverage_summary"])
display(tables["overlap_summary"])


## Input Coverage Packet
Summarize the checked-in inputs, strategy file coverage, and realized overlap window before reviewing any regime or transition diagnostics. If optional strategy metadata is absent, the coverage table preserves the strategy rows and leaves those metadata columns blank.


In [ ]:
display(tables["strategy_coverage_summary"])


## Macro Regime Summaries
Show both the full macro regime file counts and the realized Macro_Regime counts observed inside the current strategy window.


In [ ]:
display(tables["macro_regime_file_summary"])
display(tables["realized_macro_regime_category_counts"])


## Regime Diagnostics And Coverage Warnings
Reuse the existing regime diagnostics path to show descriptive per-strategy summaries and highlight sparse or missing regime coverage warnings.


In [ ]:
diagnostic_columns = [
    "strategy_id",
    "Macro_Regime",
    "obs",
    "mean_return",
    "cumulative_return",
    "volatility",
    "min_return",
    "max_return",
    "max_drawdown",
    "average_drawdown",
]
display(tables["regime_diagnostics_summary"][diagnostic_columns])
display(tables["sparse_missing_regime_warnings"])


## Transition Coverage Packet
Summarize transition coverage and clearly separate aligned transition rows from skipped rows outside the current strategy coverage window. Zero aligned transitions is acceptable when the current strategy window does not overlap any transition months and is not a failure. When aligned transitions do exist, the packet reports whether the aggregate transition summaries are based on complete per-strategy windows or whether incomplete windows, including zero-observation strategy windows, were excluded.


In [ ]:
display(tables["transition_coverage_summary"])
display(tables["transition_window_status"])
display(tables["transition_window_completeness_summary"])
display(tables["aligned_skipped_transition_summary"])
display(tables["aligned_transition_rows"])
display(tables["skipped_transition_rows"])


## Caveats And Deferred Work
Keep the packet explicit about current strategy coverage, limited realized regime coverage, and the fact that this project currently provides descriptive research infrastructure rather than a trading model. Transition-aligned aggregate summaries should be read as complete-window-only diagnostics when the packet reports incomplete strategy windows, including zero-observation gaps caused by uneven strategy calendars.


In [ ]:
display(tables["caveats"])
display(tables["deferred_work"])
